In [11]:
# Complete U-Net Segmentation with SHAP Integration

# Step 1: Install necessary libraries
!pip install tensorflow
!pip install shap
!pip install opencv-python
!pip install matplotlib

# Step 2: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import shap
import cv2
import os
import matplotlib.pyplot as plt
from google.colab import files

# Step 3: Define U-Net model
def unet_model(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    # Decoder
    u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

    u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)

    u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Step 4: Load and preprocess data
def load_data():
    uploaded = files.upload()
    images = []
    masks = []

    for filename in uploaded.keys():
        if filename.endswith('.png'):
            img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
            if 'mask' in filename:
                masks.append(img)
            else:
                images.append(img)

    images = np.array(images)
    masks = np.array(masks)

    # Normalize images
    images = images.astype('float32') / 255.0
    masks = masks.astype('float32') / 255.0

    return images, masks

# Step 5: Train the model
def train_model(model, x_train, y_train):
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=50, batch_size=16, validation_split=0.2)

# Step 6: SHAP integration for explainability
def explain_model(model, x_train):
    explainer = shap.KernelExplainer(model.predict, x_train)
    shap_values = explainer.shap_values(x_train)
    return shap_values

# Step 7: Visualize SHAP values
def visualize_shap_values(shap_values, x_train):
    shap.summary_plot(shap_values, x_train)

# Main execution
if __name__ == "__main__":
    input_shape = (128, 128, 1)  # Example input shape for MRI images
    model = unet_model(input_shape)

    # Load your data
    x_train, y_train = load_data()  # Now it will prompt for file uploads

    # Train the model
    train_model(model, x_train, y_train)

    # Generate SHAP values
    shap_values = explain_model(model, x_train)

    # Visualize SHAP values
    visualize_shap_values(shap_values, x_train)

Saving brain scan.jpg to brain scan (7).jpg


ValueError: Training data contains 0 samples, which is not sufficient to split it into a validation and training set as specified by `validation_split=0.2`. Either provide more data, or a different value for the `validation_split` argument.